# PrismaticCell — interactive cell explorer

One notebook to **configure everything**, run a **cycling protocol**, and explore the results:
time curves, 2-D heatmaps (temperature / SOC / current), per-layer maps of a chosen sandwich,
and three animatable **3-D views** of the cell.

Workflow: edit **§1 Configuration** (every model setting lives in the YAML cell) and
**§2 Cycling protocol**, then *Run All*. Re-run any time after changing settings.

> Fidelity notes: control volumes homogenize the 5-layer sandwich (T/SOC are shared by the
> layers at a location; collector potentials are per-foil). Individual sandwiches are thinner
> than a mesh cell — per-sandwich views show the mesh slice containing that sandwich.

In [ ]:
# §0 Setup ------------------------------------------------------------------
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook lives in notebooks/
sys.path.insert(0, REPO)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from prismaticcell.config import SimConfig
from prismaticcell import coupling, viz, viz3d
from prismaticcell.cycling import configure_cycling
from prismaticcell.geometry import build_geometry

%matplotlib inline
from IPython.display import display
try:
    import ipywidgets as W
    INTERACT = True
except Exception:
    INTERACT = False
# widget comms hang under headless executors (nbclient/nbconvert) -- render static
# defaults there; full sliders appear when you open the notebook live
if os.environ.get("PRISMATIC_HEADLESS"):
    INTERACT = False
print("interactive widgets:", INTERACT)

## §1 Configuration — every setting of the model

Everything is here: materials, the 5-layer sandwich (roles / materials / thicknesses), electrode
dimensions, jellyrolls & stacking axis, mylar wrap, electrolyte fill, the fixed can (outer size +
wall thickness), bottom insulator, headspace, tabs (placement / size / protrusion), terminal
thermal boundary, **mesh resolution**, per-face **boundary conditions** (convection / fixed-T /
flux / adiabatic, each with optional radiation `emissivity`), and solver controls.

Edit and re-run. Units are SI (m, K, W); faces may use role names (`front_face`, `top_face`, …).

In [ ]:
CONFIG_YAML = """
name: notebook_cell

materials:                       # thermal + electrical properties (edit or add your own)
  al_collector:   {density: 2700.0, cp: 900.0,  k_through: 238.0, k_in: 238.0, sigma_elec: 3.5e+7}
  cu_collector:   {density: 8960.0, cp: 385.0,  k_through: 398.0, k_in: 398.0, sigma_elec: 5.96e+7}
  lfp_cathode:    {density: 2200.0, cp: 1000.0, k_through: 1.0,   k_in: 1.0}
  graphite_anode: {density: 1550.0, cp: 1100.0, k_through: 1.5,   k_in: 1.5}
  separator:      {density: 1200.0, cp: 1900.0, k_through: 0.30,  k_in: 0.30}
  can_al:         {density: 2700.0, cp: 900.0,  k_through: 238.0, k_in: 238.0}
  gap_air:        {density: 1.2,    cp: 1005.0, k_through: 0.03,  k_in: 0.03}
  pp_insulator:   {density: 905.0,  cp: 1900.0, k_through: 0.20,  k_in: 0.20}
  mylar_film:     {density: 1390.0, cp: 1170.0, k_through: 0.15,  k_in: 0.15}
  electrolyte:    {density: 1200.0, cp: 2000.0, k_through: 0.60,  k_in: 0.60}

ecm:
  capacity_Ah: 200.0             # total cell capacity (placeholder -- set your value)
  ocv_table: ../data/lfp_ocv.csv
  entropy_table: ../data/lfp_entropy.csv
  r0_table: ../data/lfp_r0.csv
  ea_r0: 20000.0
  t_ref: 298.15
  rc_pairs:
    - {r_table: ../data/lfp_rc1_r.csv, c_table: ../data/lfp_rc1_c.csv, ea_r: 22000.0, ea_c: 0.0}

assembly:
  stack_axis: y                  # through-plane / stacking axis (length=X, height=Z)
  arrangement: stacked           # two jellyrolls back-to-back along Y
  inter_gap: 1.0e-3              # m, electrolyte gap between the rolls (0 = touching)
  wall_clearance: 5.0e-4         # m (used only when outer_dims is removed -> auto-sized can)
  cavity_fill: electrolyte       # flooded cell ('gap_air' for dry)
  roll_wrap:                     # mylar film on each roll's 4 side faces (remove block for none)
    material: mylar_film
    thickness: 5.0e-5            # m (50 um)
    coverage: sides              # sides | big_faces | ends | all

enclosure:
  kind: prismatic
  material: can_al
  wall_thickness: 4.0e-4         # m, fixed can wall on all faces
  outer_dims: [0.730, 0.0153, 0.130]   # fixed can OUTER size [X,Y,Z] m (remove -> auto-size)
  headspace_fill: gap_air        # gas above the electrolyte level (= roll top)
  contact_conductance: 150.0     # W/m^2K roll<->wall interfacial conductance
  wall_model: shell
  tab_heat_sink: false           # terminal NOT cooled (set true + tab_sink_t for a busbar temp)
  # tab_sink_t: 308.15           # K, busbar temperature (only with tab_heat_sink: true)
  insulator:                     # bottom insulation film on the can floor (remove for none)
    material: pp_insulator
    thickness: 5.0e-4            # m
    location: bottom

tabs:                            # weld footprints flush at the top edge; protrusion = external tab
  - {polarity: pos, loc_length: 0.15, loc_height: 0.9583, size_length: 0.05, size_height: 0.010, protrusion: 0.012}
  - {polarity: neg, loc_length: 0.85, loc_height: 0.9583, size_length: 0.05, size_height: 0.010, protrusion: 0.012}

jellyrolls_stack: &sandwich      # ONE sandwich = CC | cathode | separator | anode | CC
  width: 0.720                   # m, electrode LENGTH (X)
  height: 0.120                  # m, electrode HEIGHT (Z)
  layers:
    - {role: pos_collector,   material: al_collector,   thickness: 13.0e-6}
    - {role: cathode_coating, material: lfp_cathode,    thickness: 92.0e-6}
    - {role: separator,       material: separator,      thickness: 14.0e-6}
    - {role: anode_coating,   material: graphite_anode, thickness: 73.0e-6}
    - {role: neg_collector,   material: cu_collector,   thickness: 6.0e-6}

mesh:                            # grid resolution along physical x / y / z
  nx: 24
  ny: 10
  nz: 12

cooling:                         # per-face BCs: convection {h,t_inf} | dirichlet {t_inf} |
  front_face: {kind: convection, h: 100.0, t_inf: 298.15}   #   neumann {flux} | adiabatic;
  back_face:  {kind: convection, h: 100.0, t_inf: 298.15}   #   add emissivity: 0..1 for radiation
  left_face:  {kind: adiabatic}
  right_face: {kind: adiabatic}
  top_face:   {kind: adiabatic}
  bottom_face: {kind: adiabatic}

load: {kind: constant_crate, value: 1.0}   # overwritten by the cycling protocol below

solver:
  mode: transient
  dt: 60.0                       # s, timestep
  t_end: 3600.0                  # overwritten by the cycling protocol
  collector_model: planar        # planar (fast) | layered (full 3-D collector)

t_init: 298.15                   # K, initial temperature
"""

# expand the sandwich anchor into the two jellyrolls, then write + load
import yaml as _yaml
_raw = _yaml.safe_load(CONFIG_YAML)
_sw = _raw.pop("jellyrolls_stack")
N_STACKS_PER_ROLL = 34            # sandwiches per jellyroll  <-- edit
_raw["assembly"]["jellyrolls"] = [{"n_stacks": N_STACKS_PER_ROLL, "stack": _sw},
                                  {"n_stacks": N_STACKS_PER_ROLL, "stack": _sw}]
CFG_PATH = os.path.join(REPO, "configs", "_notebook_cell.yaml")
with open(CFG_PATH, "w") as fh:
    _yaml.safe_dump(_raw, fh, sort_keys=False)
cfg = SimConfig.from_yaml(CFG_PATH)
geom = build_geometry(cfg)
print(f"cell OK: can {tuple(round(d*1e3,1) for d in geom.outer_dims)} mm, "
      f"{sum(r.n_stacks for r in geom.rolls)} sandwiches, "
      f"{geom.n_active} active control volumes, mesh {geom.grid.nx}x{geom.grid.ny}x{geom.grid.nz}")

## §2 Cycling protocol

CC discharge/charge over an SOC window. The mean SOC traverses `[SOC_MIN, SOC_MAX]` exactly at
the given C-rate; individual control volumes spread around it (a real effect — see the SOC maps).

In [ ]:
C_RATE    = 1.0      # discharge/charge rate [C]
N_CYCLES  = 1        # full cycles (discharge + charge)
SOC_MIN   = 0.2
SOC_MAX   = 0.8
START     = "discharge"   # or "charge"
DT        = 60.0     # s, timestep

configure_cycling(cfg, c_rate=C_RATE, n_cycles=N_CYCLES, soc_min=SOC_MIN, soc_max=SOC_MAX,
                  start=START, dt=DT)
print(f"protocol: {N_CYCLES} cycle(s) x ({SOC_MAX}->{SOC_MIN}) at {C_RATE}C "
      f"= {cfg.solver.t_end:.0f} s total, {int(cfg.solver.t_end/cfg.solver.dt)} steps")

# preview the current profile
_t = np.linspace(0, cfg.solver.t_end, 600)
_i = [coupling.applied_at(cfg, tt)[1] for tt in _t]
fig, ax = plt.subplots(figsize=(9, 2.2))
ax.plot(_t/60, _i, lw=1.5, color="#31708E")
ax.set_xlabel("time [min]"); ax.set_ylabel("current [A]\n(+ = discharge)")
ax.grid(alpha=0.3); fig.tight_layout()

## §3 Run

Two-way coupled electro-thermal transient. Every step solves the collector network (one ECM per
control volume), forms the heat map (ECM + foil ohmic + tab Joule backflow), and advances the 3-D
temperature field; SOC/current field histories are recorded for the animations.

In [ ]:
res = coupling.run(cfg)
print(f"steps: {len(res.t)}   V: {res.v_terminal.min():.3f}..{res.v_terminal.max():.3f} V")
print(f"mean SOC: {res.soc_mean.min():.3f}..{res.soc_mean.max():.3f}")
print(f"T: mean {res.T_mean[-1]-273.15:.2f} C  max {res.T_max.max()-273.15:.2f} C")
print(f"energy closure: {res.energy_balance['closure_rel']:.2e}   (~1e-12 = machine precision)")
NT = len(res.t)

## §4 Time curves — current, SOC, voltage (+ temperature)

In [ ]:
_ = viz.plot_time_series(res)

## §5 Heatmaps on the electrode plane — temperature / SOC / current

Slide through time; colors are fixed over the whole run so frames are comparable. (The tab stubs
on the temperature view are the 1-D fin profile — the tab is the same foil continuing out.)

In [ ]:
def _plane(field="T", ti=NT-1):
    fig = viz.plot_plane_field(res, field=field, ti=ti)
    display(fig); plt.close(fig)

if INTERACT:
    W.interact(_plane, field=["T", "soc", "current"],
               ti=W.IntSlider(value=NT-1, min=0, max=NT-1, step=1, description="time idx"))
else:
    for f in ("T", "soc", "current"):
        fig = viz.plot_plane_field(res, field=f, ti=NT-1); display(fig); plt.close(fig)

In [ ]:
# per-collector view (one tab per foil) + potential contours, at the final time
_ = viz.plot_collector_planes(res)

## §6 Per-layer maps of a specific sandwich

Pick a sandwich number (1..68 across both rolls, in stacking order) and a layer — e.g. the anode
current collector of sandwich #5. Collector layers show their foil's potential drop (the in-plane
current driver); coatings/separator show the through-cell current density.

In [ ]:
N_SANDWICH = sum(r.n_stacks for r in res.geom.rolls)
_LAYERS = ["pos_collector", "cathode_coating", "separator", "anode_coating", "neg_collector"]

def _layer_maps(sandwich=5, layer="neg_collector", ti=NT-1):
    fig = viz.plot_stack_layer_maps(res, sandwich=sandwich, layer=layer, ti=ti)
    display(fig); plt.close(fig)

if INTERACT:
    W.interact(_layer_maps,
               sandwich=W.IntSlider(value=5, min=1, max=N_SANDWICH, step=1),
               layer=_LAYERS,
               ti=W.IntSlider(value=NT-1, min=0, max=NT-1, step=1, description="time idx"))
else:
    fig = viz.plot_stack_layer_maps(res, sandwich=5, layer="neg_collector", ti=NT-1)
    display(fig); plt.close(fig)

## §7 3-D views (animatable)

Three views, each colored by temperature / SOC / current at the chosen time (the thin thickness
axis is drawn exaggerated; axes keep true mm):
1. **Full cell — exterior only**
2. **Full cell — cutaway** (front part removed at an adjustable fraction of the length, exposing
   the internal cross-section; wireframe = the real can outer envelope)
3. **The two jellyrolls** (pulled apart along the stacking axis for visibility)

Drag the time slider to animate; §8 saves GIFs.

In [ ]:
def _ext(ti=0, field="T"):
    fig = viz3d.plot_cell_3d(res, ti=ti, field=field)
    display(fig); plt.close(fig)

if INTERACT:
    W.interact(_ext, ti=W.IntSlider(0, 0, NT-1, 1, description="time idx"),
               field=["T", "soc", "current"])
else:
    fig = viz3d.plot_cell_3d(res, ti=0, field="T"); display(fig); plt.close(fig)

In [ ]:
def _cut(ti=0, field="T", cut_frac=0.5):
    fig = viz3d.plot_cell_cutaway_3d(res, ti=ti, field=field, cut_frac=cut_frac)
    display(fig); plt.close(fig)

if INTERACT:
    W.interact(_cut, ti=W.IntSlider(0, 0, NT-1, 1, description="time idx"),
               field=["T", "soc", "current"],
               cut_frac=W.FloatSlider(0.5, min=0.1, max=0.9, step=0.1))
else:
    fig = viz3d.plot_cell_cutaway_3d(res, ti=0, field="T"); display(fig); plt.close(fig)

In [ ]:
def _rolls(ti=0, field="T"):
    fig = viz3d.plot_jellyrolls_3d(res, ti=ti, field=field)
    display(fig); plt.close(fig)

if INTERACT:
    W.interact(_rolls, ti=W.IntSlider(0, 0, NT-1, 1, description="time idx"),
               field=["T", "soc", "current"])
else:
    fig = viz3d.plot_jellyrolls_3d(res, ti=0, field="T"); display(fig); plt.close(fig)

## §8 Save animations (GIF)

Set `RUN_ANIMATIONS = True` and re-run this cell. Writes GIFs under `outputs/` and shows them
inline. `stride` skips time steps; higher = faster to render.

In [ ]:
RUN_ANIMATIONS = False
if RUN_ANIMATIONS:
    from IPython.display import Image
    outdir = os.path.join(REPO, "outputs"); os.makedirs(outdir, exist_ok=True)
    for view, tag in ((viz3d.plot_cell_3d, "exterior"),
                      (viz3d.plot_cell_cutaway_3d, "cutaway"),
                      (viz3d.plot_jellyrolls_3d, "jellyrolls")):
        p = viz3d.save_animation(res, view, field="T",
                                 path=os.path.join(outdir, f"anim_{tag}_T.gif"), stride=4)
        print("wrote", p)
        display(Image(p))